In [2]:
import pandas as pd 
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics  import classification_report , accuracy_score
import numpy as np


In [4]:
df = pd.read_csv('/home/aadrika.sharma/fastapi-demo-api/Serving_ML_Models_with_fastAPI/insurance.csv')

In [5]:
df.sample(5)

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category
96,26,113.8,1.54,34.010000,False,Delhi,private_job,Low
77,67,114.5,1.74,0.610000,True,Mumbai,retired,High
14,49,89.3,1.65,13.505166,False,Kota,government_job,Medium
36,61,58.4,1.64,0.530000,False,Hyderabad,retired,Medium
88,54,97.4,1.76,30.000000,False,Mumbai,government_job,Low


In [6]:
df['occupation'].unique()

array(['retired', 'freelancer', 'student', 'government_job',
       'business_owner', 'unemployed', 'private_job'], dtype=object)

In [7]:
df_feat = df.copy()

In [8]:
#Feature 1 :BMI

df_feat["bmi"] = df_feat["weight"]/(df_feat["height"]**2)


In [9]:
#Feature 2 :Age Group

def age_group(age):
    if age <18:
        return "young"
    elif age >=18 and age < 60:
        return "adult"
    else:
        return "senior"

In [10]:
df_feat["age_group"] = df_feat["age"].apply(age_group)


In [11]:
#Feature 3:Life Style Risk

def lifestyle_risk(row):
    if row["smoker"] and row["bmi"] > 30:
        return "high"
    elif row["smoker"] or row["bmi"] > 30:
        return "medium"
    else:
        return "low"

In [12]:
df_feat["lifestyle_risk"] = df_feat.apply(lifestyle_risk, axis=1)

In [13]:
tier_1_cities=["Mumbai", "Delhi", "Bangalore", "Hyderabad", "Ahmedabad", "Chennai", "Kolkata", "Pune"
               ]

tier_2_cities = ["Jaipur", "Lucknow", "Kanpur", "Nagpur", "Indore", "Thane", "Bhopal", "Visakhapatnam", "Pimpri-Chinchwad",]


In [14]:
#Feature 4:City Tier
def city_tier(city):
    if city in tier_1_cities:
        return 1
    elif city in tier_2_cities:
        return 2
    else:
        return 3

In [ ]:
df_feat["city_tier"] = df_feat["city"].apply(city_tier)

In [19]:
df_feat.drop(
    columns=['age', 'weight', 'height', 'smoker', 'city']
)[
    [
        'income_lpa',
        'occupation',
        'bmi',
        'age_group',
        'lifestyle_risk',
        'insurance_premium_category'
    ]
].sample(5)

,income_lpa,occupation,bmi,age_group,lifestyle_risk,insurance_premium_category
40,40.19,unemployed,1,adult,medium,Medium
38,1.84,retired,2,senior,high,High
88,30.00,government_job,1,adult,medium,Low
23,23.71,unemployed,3,adult,low,Medium
51,28.95,private_job,2,adult,high,High


In [18]:
print(df_feat.columns.tolist())

['age', 'weight', 'height', 'income_lpa', 'smoker', 'city', 'occupation', 'insurance_premium_category', 'bmi', 'age_group', 'lifestyle_risk']


In [ ]:
#Select features and target variable
X